# Solución P02: Parquet, contrato y reejecución

[Enunciado](../talleres/P02.md) · [Índice](README.md)

Ejecutar de arriba abajo con el kernel del entorno `.venv` preparado en la [guía WSL](../docs/instalacion-wsl.md). Usa las tres muestras incluidas; no descarga los archivos completos. Las salidas propias se regeneran al repetir la ejecución.

In [ ]:
CODIGO = "P02"
from pathlib import Path
from types import SimpleNamespace
import sys, json
import pandas as pd
from IPython.display import display

# Funciona desde soluciones/, la raíz del repositorio o la carpeta base bigdata.
roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'big-data-postgraduate']
REPO = next((p for p in roots if (p / 'kit/pqrs_talleres.py').is_file()), None)
assert REPO is not None, 'Abrir Jupyter desde bigdata/big-data-postgraduate o soluciones/'
KIT = REPO / 'kit'
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
OUT = KIT / 'salidas/soluciones' / CODIGO
OUT.mkdir(parents=True, exist_ok=True)
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=OUT, repeticiones=3)
print('Repositorio:', REPO, '\nSalidas:', OUT)


## Tareas 1 y 2. Publicación trazable
El procedimiento verifica primero los hashes y conserva los CSV originales. La proyección tiene 16 columnas, frente a las 38 originales; `corte_archivo` y `fila_fuente` permiten volver al origen.

In [ ]:
primera=lab.formatos(a)
lab.guardar(OUT/'primera_ejecucion.json',primera)
import pyarrow.parquet as pq
tabla=pq.read_table(OUT/'pqrs.parquet')
assert tabla.num_rows==3000 and tabla.num_columns==16
display(tabla.schema)
display(tabla.select(['corte_archivo','fila_fuente','OBJECTID']).slice(0,5).to_pandas())

## Tarea 3. Archivo simple y particiones
La partición procede de `periodo` y `mes`, no del semestre del nombre de archivo. Conciliamos ambos productos y mostramos el costo de almacenamiento.

In [ ]:
import duckdb
with duckdb.connect() as con:
    filas=con.execute('SELECT count(*) FROM read_parquet(?)',[str(OUT/'pqrs_particionado/**/*.parquet')]).fetchone()[0]
assert filas==tabla.num_rows
partes=list((OUT/'pqrs_particionado').rglob('*.parquet'))
assert len(partes)==8
display(pd.DataFrame({'archivo':[str(p.relative_to(OUT)) for p in partes], 'bytes':[p.stat().st_size for p in partes]}))
print('Bytes simple:',(OUT/'pqrs.parquet').stat().st_size,'; particionado:',sum(p.stat().st_size for p in partes))

## Tarea 4. Reejecución
Repetir reemplaza los productos, no agrega filas. La firma comprueba estabilidad del control seleccionado; no demuestra igualdad de todas las propiedades semánticas.

In [ ]:
segunda=lab.formatos(a)
for key in ['filas','firma_contenido','columnas_producto']:
    assert primera[key]==segunda[key], key
lab.guardar(OUT/'segunda_ejecucion.json',segunda)
display(pd.DataFrame([primera,segunda],index=['primera','segunda']))

## Tarea 5. Fallo de contrato en memoria
Retiramos una columna de una copia. El contrato debe impedir la publicación; el archivo original no se modifica.

In [ ]:
f,p=lab.seleccion(a)[0]
original=next(lab.leer(f,p,100))
defectuoso=original.drop(columns=['mes'])
try:
    lab.normalizar(defectuoso,f,0)
except ValueError as error:
    incidencia={'columna_retirada':'mes','error':str(error),'accion':'detener publicación'}
else:
    raise AssertionError('El contrato no detectó la columna ausente')
lab.guardar(OUT/'incidencia_contrato.json',incidencia)
restaurado=lab.normalizar(original,f,0)
assert restaurado.num_rows==100
display(incidencia)
lab.guardar(OUT/'configuracion_solucion.json',{'bloque':a.bloque,'completo':False})

**Decisión:** publicar solo tras verificar origen, contrato y conciliación. Restaurar la columna elimina este fallo concreto, pero no reemplaza los controles semánticos. Particionar puede producir archivos pequeños y empeorar el rendimiento; no implica una mejora automática. La entrega incluye controles de ambas ejecuciones, configuración e incidencia.